#### General guidance

This serves as a template which will guide you through the implementation of this task. It is advised
to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# Add any additional imports here (however, the task is solvable without using 
# any additional imports)
# import ...
from sklearn.linear_model import Ridge

 #### Loading data

In [2]:
data = pd.read_csv("train.csv")
y = data["y"].to_numpy()
data = data.drop(columns="y")
# print a few data samples
print(data.head())

         x1   x2     x3   x4     x5     x6     x7      x8    x9    x10   x11  \
0   0.06724  0.0   3.24  0.0  0.460  6.333   17.2  5.2146   4.0  430.0  16.9   
1   9.23230  0.0  18.10  0.0  0.631  6.216  100.0  1.1691  24.0  666.0  20.2   
2   0.11425  0.0  13.89  1.0  0.550  6.373   92.4  3.3633   5.0  276.0  16.4   
3  24.80170  0.0  18.10  0.0  0.693  5.349   96.0  1.7028  24.0  666.0  20.2   
4   0.05646  0.0  12.83  0.0  0.437  6.232   53.7  5.0141   5.0  398.0  18.7   

      x12    x13  
0  375.21   7.34  
1  366.15   9.53  
2  393.74  10.50  
3  396.90  19.77  
4  386.40  12.34  


#### Calculating the average RMSE

In [3]:
def calculate_RMSE(w, X, y):
    """This function takes test data points (X and y), and computes the empirical RMSE of 
    predicting y from X using a linear model with weights w. 

    Parameters
    ----------
    w: array of floats: dim = (13,), optimal parameters of ridge regression 
    X: matrix of floats, dim = (15,13), inputs with 13 features
    y: array of floats, dim = (15,), input labels

    Returns
    ----------
    RMSE: float: dim = 1, RMSE value
    """
    RMSE = 0
    # TODO: Enter your code here
    y_pred = np.dot(X,w)
    RMSE = mean_squared_error(y,y_pred, squared=False) #squared = FALSE ensures the root of the MSE is taken
    #RMSE2 = np.sqrt(mean_squared_error(y,y_pred)) #sanity check: should return the same as the above - is true
    #print("RMSE1",RMSE)
    #print("RMSE2", RMSE2)
    assert np.isscalar(RMSE)
    return RMSE

#### Fitting the regressor

In [4]:
def fit(X, y, lam):
    """
    This function receives training data points, then fits the ridge regression on this data
    with regularization hyperparameter lambda. The weights w of the fitted ridge regression
    are returned. 

    Parameters
    ----------
    X: matrix of floats, dim = (135,13), inputs with 13 features
    y: array of floats, dim = (135,), input labels)
    lam: float. lambda parameter, used in regularization term

    Returns
    ----------
    w: array of floats: dim = (13,), optimal parameters of ridge regression
    """
    w = np.zeros((13,))
    # TODO: Enter your code here
    # uses library functions but could potentially try implementing ourselves? But that's probably not going to end well
    ridge_reg = Ridge(alpha=lam,fit_intercept = False) #as we only pass the coeffs/weights, we need the bias b to be 0 
    w = ridge_reg.fit(X,y).coef_
    
    assert w.shape == (13,)
    return w

#### Performing computation

In [8]:
"""
Main cross-validation loop, implementing 10-fold CV. In every iteration 
(for every train-test split), the RMSE for every lambda is calculated, 
and then averaged over iterations.

Parameters
---------- 
X: matrix of floats, dim = (150, 13), inputs with 13 features
y: array of floats, dim = (150, ), input labels
lambdas: list of floats, len = 5, values of lambda for which ridge regression is fitted and RMSE estimated
n_folds: int, number of folds (pieces in which we split the dataset), parameter K in KFold CV

Compute
----------
avg_RMSE: array of floats: dim = (5,), average RMSE value for every lambda
"""
X = data.to_numpy()
# The function calculating the average RMSE
lambdas = [0.1, 1, 10, 100, 200]
n_folds = 10

RMSE_mat = np.zeros((n_folds, len(lambdas)))

# TODO: Enter your code here. Hint: Use functions 'fit' and 'calculate_RMSE' with training and test data
# and fill all entries in the matrix 'RMSE_mat'
kf  = KFold(n_splits = n_folds,random_state = None)
for i,lam in enumerate(lambdas): 
    for j, (train_index, test_index) in enumerate (kf.split(X)): 
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
    
        wi = fit(X_train,y_train,lam)
        RMSE_mat[j][i] = calculate_RMSE(wi, X_test, y_test)
print(RMSE_mat)
avg_RMSE = np.mean(RMSE_mat, axis=0) # avg_RMSE: array of floats: dim = (5,), average RMSE value for every lambda
assert avg_RMSE.shape == (5,)
print(avg_RMSE) #oddly high 

[[7.4412339  7.47793331 7.58146902 8.19645872 8.50748161]
 [5.12826602 4.88393133 4.45282503 3.55256057 3.60399295]
 [7.70764701 7.70279186 7.72774439 7.77994411 7.88968326]
 [4.54006065 4.50059526 4.33989198 4.94678494 5.24054741]
 [4.07531646 4.0726219  4.19425423 4.89426434 5.24272243]
 [5.10975443 5.15192098 5.39446252 7.11982551 7.87880031]
 [6.55136404 6.56151161 6.55275056 7.13545034 7.47865912]
 [6.04021985 6.09837022 6.36717035 7.59224397 7.99355906]
 [4.88759828 4.88177704 4.70584949 4.30299719 4.45025962]
 [3.5549224  3.47254925 3.38243796 3.79878164 4.14775924]]
[5.5036383  5.48040028 5.46988555 5.93193113 6.2433465 ]


In [7]:
# Save results in the required format
np.savetxt("./results.csv", avg_RMSE, fmt="%.12f")